In [ ]:
import os
import subprocess
import re
import json
import hashlib
from datetime import datetime
import xml.etree.ElementTree as ET

# ─── CẤU HÌNH ────────────────────────────────────────────────
SITE_BASE_URL = "https://kinhnikaya.org"   # ← đổi thành domain thực của bạn
SITEMAP_CHANGEFREQ = "weekly"           # always | hourly | daily | weekly | monthly | yearly | never
SITEMAP_PRIORITY = "0.7"                 # 0.0 → 1.0
CACHE_FILE = '../docs/.sitemap-cache.json'
# ─────────────────────────────────────────────────────────────

def extract_base_path(file_path):
    dir_path = os.path.dirname(file_path)
    dir_path = dir_path.replace("\\", "/")
    dir_path = re.sub(r'^\.{0,2}/docs', '', dir_path)
    if not dir_path.startswith('/'):
        dir_path = '/' + dir_path
    if not dir_path.endswith('/'):
        dir_path = dir_path + '/'
    return dir_path

def parse_items_from_file(file_path):
    """Trả về list of { text, link, sources } từ một file .js"""
    if not os.path.exists(file_path):
        print(f"⚠️  Không tìm thấy file: {file_path}")
        return []

    base_path = extract_base_path(file_path)
    items = []

    pattern_type1 = re.compile(
        r'\{\s*text\s*:\s*(["\'])(.*?)\1\s*,\s*link\s*:\s*(["\'])(.*?)\3\s*\}'
    )
    pattern_type2_array = re.compile(r'=\s*(\[[\s\S]*?\]);', re.MULTILINE)

    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    # --- Loại 2 (params.slug) ---
    array_match = pattern_type2_array.search(content)
    found_type2 = False

    if array_match:
        try:
            data = json.loads(array_match.group(1))
            count = 0
            for entry in data:
                params = entry.get("params", {})
                slug = params.get("slug", "")
                title = params.get("data", {}).get("title", "")

                # Trích xuất nguồn file .md để tính băm (hash)
                left = params.get("data", {}).get("left", "")
                right = params.get("data", {}).get("right", "")
                sources = []
                if left:
                    sources.append(os.path.normpath(os.path.join('../docs', left.lstrip('/'))))
                if right:
                    sources.append(os.path.normpath(os.path.join('../docs', right.lstrip('/'))))
                if not sources:
                    sources.append(file_path)

                if slug and title:

                    items.append({"text": title, "link": base_path + slug, "sources": sources})
                    count += 1
            if count > 0:
                found_type2 = True
                print(f"✅ [Loại 2] {file_path} → base: '{base_path}' ({count} items)")
        except json.JSONDecodeError:
            pass

    # --- Loại 1 ({ text, link }) ---
    if not found_type2:
        matches = pattern_type1.findall(content)
        for match in matches:
            link_value = match[3]
            link_path = link_value
            if link_value.endswith('.md'):
                link_path = link_value[:-3]
                link_value = link_path

            # Suy luận file nguồn
            source_file = os.path.normpath(os.path.join('../docs', link_path.lstrip('/') + '.md'))
            items.append({"text": match[1], "link": link_value, "sources": [source_file]})
        print(f"✅ [Loại 1] {file_path} ({len(matches)} items)")

    return items

def write_search_file(all_items, output_file):
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write('export const searchItems = [\n')
        for i, item in enumerate(all_items):
            safe_text = item["text"].replace('"', '\\"')
            line = f'    {{ text: "{safe_text}", link: "{item["link"]}" }}'
            if i < len(all_items) - 1:
                line += ','
            f.write(line + '\n')
        f.write('];\n')
    print(f"📄 Search file  → '{output_file}' ({len(all_items)} items)")

# def get_file_hash(filepath):
#     """Tính mã băm MD5 của một file."""
#     if not os.path.exists(filepath):
#         return ""
#     with open(filepath, 'rb') as f:
#         return hashlib.md5(f.read()).hexdigest()

# def get_item_hash(sources):
#     """Tính mã băm kết hợp cho tất cả các file nguồn của một item."""
#     combined_hash = ""
#     for src in sources:
#         combined_hash += get_file_hash(src)
#     if combined_hash:
#         return hashlib.md5(combined_hash.encode('utf-8')).hexdigest()
#     return ""


def get_git_lastmods(repo_path='../'):
    """Lấy ngày cập nhật cuối cùng của tất cả các file từ Git."""
    print("Đang lấy thông tin lịch sử Git...")
    lastmods = {}
    try:
        import subprocess
        result = subprocess.run(
            ['git', '--no-pager', 'log', '--format=COMMIT %cI', '--name-only', '--diff-filter=AM'],
            cwd=repo_path,
            capture_output=True,
            text=True,
            check=True
        )
        current_date = None
        for line in result.stdout.splitlines():
            line = line.strip()
            if not line:
                continue
            if line.startswith('COMMIT '):
                current_date = line.split(' ', 1)[1][:10]
            elif current_date:
                norm_path = os.path.normpath(os.path.join(repo_path, line))
                if norm_path not in lastmods:
                    lastmods[norm_path] = current_date
    except Exception as e:
        print(f"⚠️ Lỗi khi chạy git log: {e}")
    print (lastmods)
    return lastmods

def write_sitemap_file(all_items, output_file, base_url, changefreq, priority):
    today = datetime.now().strftime("%Y-%m-%d")
    base_url = base_url.rstrip('/')

    # 1. Đọc cache cũ nếu có
    cache = {}
    if os.path.exists(CACHE_FILE):
        try:
            with open(CACHE_FILE, 'r', encoding='utf-8') as f:
                cache = json.load(f)
        except:
            pass

    # 2. Đọc sitemap cũ để lấy lastmod fallback
    old_lastmods = {}
    if os.path.exists(output_file):
        try:
            tree = ET.parse(output_file)
            root = tree.getroot()
            ns = {'ns': 'http://www.sitemaps.org/schemas/sitemap/0.9'}
            for url in root.findall('ns:url', ns):
                loc = url.find('ns:loc', ns)
                lastmod = url.find('ns:lastmod', ns)
                if loc is not None and lastmod is not None:
                    old_lastmods[loc.text] = lastmod.text
        except Exception:
            pass

    new_cache = {}
    lines = ['<?xml version="1.0" encoding="UTF-8"?>']
    lines.append('<urlset xmlns="http://www.sitemaps.org/schemas/sitemap/0.9">')
    updated_count = 0
    git_lastmods = get_git_lastmods('../')

    for item in all_items:
        link = item["link"]

        if not link.startswith('/'):
            link = '/' + link
        full_url = base_url + link
        # print(link) # commented out to reduce noise

        # Tính hash hiện tại từ các file markdown gốc
        sources = item.get("sources", [])
        # current_hash = get_item_hash(sources)

        # Tìm ngày cập nhật mới nhất từ Git
        item_git_lastmod = None
        for src in sources:
            src_lastmod = git_lastmods.get(src)
            if src_lastmod:
                if item_git_lastmod is None or src_lastmod > item_git_lastmod:
                    item_git_lastmod = src_lastmod

        # Xác định lastmod
        cached_info = cache.get(full_url, {})

        if item_git_lastmod:
            # Ưu tiên lấy từ Git
            lastmod = max('2026-05-03', item_git_lastmod)
        else:
            lastmod = '2026-05-03'
            updated_count += 1



        lines.append('  <url>')
        lines.append(f'    <loc>{full_url}</loc>')
        lines.append(f'    <lastmod>{lastmod}</lastmod>')
        lines.append(f'    <changefreq>{changefreq}</changefreq>')
        lines.append(f'    <priority>{priority}</priority>')
        lines.append('  </url>')

    lines.append('</urlset>')

    with open(output_file, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines) + '\n')

    with open(CACHE_FILE, 'w', encoding='utf-8') as f:
        json.dump(new_cache, f, indent=2)

    print(f"🗺️  Sitemap file → '{output_file}' ({len(all_items)} URLs, {updated_count} URLs cập nhật mới)")

def merge_js_files(input_files, search_output, sitemap_output):
    all_items = []
    for file_path in input_files:
        all_items.extend(parse_items_from_file(file_path))

    write_search_file(all_items, search_output)
    write_sitemap_file(
        all_items,
        sitemap_output,
        base_url=SITE_BASE_URL,
        changefreq=SITEMAP_CHANGEFREQ,
        priority=SITEMAP_PRIORITY,
    )

    print(f"\n🎉 HOÀN TẤT! Tổng {len(all_items)} items → 2 files được tạo.")

input_files = [
    '../docs/kinhtruongbo/c-pali-tmc-vi/tmc.js',
    '../docs/kinhtruongbo/c-sujato-tmc-vi/tmc.js',
    '../docs/kinhtrungbo/c-pali-tmc-vi/tmc.js',
    '../docs/kinhtrungbo/c-nm-tmc-vi/tmc.js',
    '../docs/kinhtangchi/c-sujato-tmc-vi/tmc.js',
    '../docs/kinhtuongung/c-sujato-tmc-vi/tmc.js',
]

search_file = '../docs/search-items.js'
sitemap_file = "../docs/public/sitemap.xml"

merge_js_files(
    input_files=input_files,
    search_output=search_file,
    sitemap_output=sitemap_file,
)


✅ [Loại 2] ../docs/kinhtruongbo/c-pali-tmc-vi/tmc.js → base: '/kinhtruongbo/c-pali-tmc-vi/' (34 items)
✅ [Loại 2] ../docs/kinhtruongbo/c-sujato-tmc-vi/tmc.js → base: '/kinhtruongbo/c-sujato-tmc-vi/' (34 items)
✅ [Loại 2] ../docs/kinhtrungbo/c-pali-tmc-vi/tmc.js → base: '/kinhtrungbo/c-pali-tmc-vi/' (152 items)
✅ [Loại 2] ../docs/kinhtrungbo/c-nm-tmc-vi/tmc.js → base: '/kinhtrungbo/c-nm-tmc-vi/' (152 items)
✅ [Loại 2] ../docs/kinhtangchi/c-sujato-tmc-vi/tmc.js → base: '/kinhtangchi/c-sujato-tmc-vi/' (186 items)
✅ [Loại 2] ../docs/kinhtuongung/c-sujato-tmc-vi/tmc.js → base: '/kinhtuongung/c-sujato-tmc-vi/' (56 items)
📄 Search file  → '../docs/search-items.js' (614 items)
Đang lấy thông tin lịch sử Git...
{'../.scripts/4make_compare_list.ipynb': '2026-05-17', '../.scripts/5sitemap-search-file.ipynb': '2026-05-17', '../docs/.sitemap-cache.json': '2026-05-17', '../docs/.vitepress/compare-render.js': '2026-05-17', '../docs/.vitepress/components/TextCompare.vue': '2026-05-17', '../docs/.vitep